# FAISS indexing

Goal:
- load saved embeddings for each chunk size
- normalize embedding vectors
- build one FAISS index per chunk size
- save indexes to disk for later retrieval experiments

Important:
- FAISS indexes are built once per chunk size
- the same indexes will later be reused for all top-k retrieval runs
- exact search is used to keep the experiment controlled and interpretable

In [1]:
import json
import faiss
import numpy as np
from pathlib import Path

data_dir = Path("/home/a/arfaoui/rag_project/data")
chunk_sizes = [32, 128, 256]

In [2]:
# Load saved embeddings and metadata for each chunk size
# We keep both because FAISS needs vectors, and retrieval later needs metadata mapping.

all_embeddings = {}
all_metadata = {}

for size in chunk_sizes:
    # Load embedding matrix from .npy file
    emb_path = data_dir / f"embeddings_{size}.npy"
    embeddings = np.load(emb_path)
    all_embeddings[size] = embeddings

    # Load chunk metadata from JSON file
    meta_path = data_dir / f"chunks_metadata_{size}.json"
    with open(meta_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    all_metadata[size] = metadata

    # Print quick sanity checks
    print(f"\nchunk_size={size}")
    print(f"Embeddings shape: {embeddings.shape}")
    print(f"Metadata entries: {len(metadata)}")


chunk_size=32
Embeddings shape: (22186, 384)
Metadata entries: 22186

chunk_size=128
Embeddings shape: (7295, 384)
Metadata entries: 7295

chunk_size=256
Embeddings shape: (5236, 384)
Metadata entries: 5236


In [13]:
# Check if embeddings are normalized or not 

sample = all_embeddings[128][:10]
all = all_embeddings[128]
sample_check = np.linalg.norm(sample, axis=1)
norms = np.linalg.norm(all, axis=1)

print(sample_check)
print(f"Mean norm all embeddings: {np.mean(norms):.4f}")
print(f"Min norm all embeddings: {np.min(norms):.4f}")
print(f"Max norm all embeddings: {np.max(norms):.4f}")

[0.99999994 0.9999999  1.         0.99999994 1.         0.99999994
 0.99999994 0.99999994 1.         1.        ]
Mean norm all embeddings: 1.0000
Min norm all embeddings: 1.0000
Max norm all embeddings: 1.0000


**Observations**
- From the inspection, the embeddings appear to be normalized as the overall min, max, and mean values across all embedding vectors are approximately 1.0.
- Normalization is a crucial step. Skipping it introduces hidden bias if embeddings were not normalized:
  • Vectors with larger magnitudes will dominate the similarity scores  
  • Retrieval quality degrades because comparisons become scale‑dependent

Note:
 - all embedings (32 and 256) are checked and all normalized 
